In [1]:
# Version X
# should calculate with float64 for precision
import json
import pandas as pd
import gc
from pathlib import Path
import numpy as np

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")

# The t-ohlcv data was pre-fetched
folder_path = find_project_root() / "data" / "bigData"

symbol = "BTCUSDT"
tf = "5m"

PATH_5m =folder_path / f"{symbol}-regimed-{tf}.json"

with open(PATH_5m) as f:
    arr = json.load(f)

# Binance API fetched columns
cols = ["timestamp", "open", "high", "low", "close", "vol", "close-timestamp", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol","checksum"]

df_5 = pd.DataFrame(arr, columns=cols)
del arr  
gc.collect() # free the Python var, preserve RAM

# Halve memory: float64 -> float32 for price/volume columns
df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]] = df_5[["open", "high", "low", "close", "vol", "quote-vol", "taker#", "taker-buy-basevol","taker-buy-quotevol"]].astype("float32")

print(f"5m Shape: {df_5.shape} | Memory: {df_5.memory_usage(deep=True).sum()/ 1e6:.2f} MB") # 160 MB

# sort by timestamp
df_5 = df_5.sort_values('timestamp').reset_index(drop=True)
df_5.head()

5m Shape: (546454, 12) | Memory: 55.74 MB


,timestamp,open,high,low,close,vol,close-timestamp,quote-vol,taker#,taker-buy-basevol,taker-buy-quotevol,checksum
0,1609187400000,26852.179688,26873.220703,26688.109375,26736.640625,598.524536,1609187699999,16039656.0,9027.0,199.124802,5336880.00,0
1,1609187700000,26735.859375,26796.000000,26678.000000,26763.359375,527.531250,1609187999999,14105745.0,9375.0,215.260605,5755750.50,0
2,1609188000000,26763.359375,26824.490234,26733.599609,26779.490234,320.595581,1609188299999,8584349.0,5462.0,131.724228,3527292.00,0
3,1609188300000,26779.480469,26799.919922,26702.000000,26744.679688,254.103180,1609188599999,6797498.0,5055.0,108.755074,2909258.25,0
4,1609188600000,26744.970703,26861.980469,26744.970703,26850.050781,243.299011,1609188899999,6523452.5,4200.0,131.684021,3530466.00,0


# Does price hit ±ATR42 within the next 3 bars?
- prepare ATR42 on 5m 
- used to make triple barrier

In [2]:
# ATR42 on 5m — equivalent to ATR14 on 15m in time

prev_close_5m = df_5['close'].shift(1)

# True range
tr_5m = pd.concat([
    df_5['high'] - df_5['low'],
    (df_5['high'] - prev_close_5m).abs(),
    (df_5['low']  - prev_close_5m).abs(),
], axis=1).max(axis=1)

# Average True Range : SMA or RMA
df_5['atr42'] = tr_5m.rolling(42).mean().astype('float32')

print(f"ATR42 range: {df_5['atr42'].min():.2f} – {df_5['atr42'].max():.2f}")
df_5[['timestamp', 'close', 'atr42']].head()

ATR42 range: 1.58 – 1552.81


,timestamp,close,atr42
0,1609187400000,26736.640625,NaN
1,1609187700000,26763.359375,NaN
2,1609188000000,26779.490234,NaN
3,1609188300000,26744.679688,NaN
4,1609188600000,26850.050781,NaN


# Triple Barrier
- Create ternary label on 5m

In [4]:
# triple-barrier on 5m (gap-aware)
FIVE_MIN_MS = 300_000

# Snap 5m timestamps to grid (absorbs ms jitter) — needed for gap detection
# df_5['ts_5m'] = (df_5['timestamp'] // FIVE_MIN_MS) * FIVE_MIN_MS

# Triple-barrier on 5m (gap-aware)
ATR_MULT    = 1.0
MAX_BARS_5M = 3   # 1h lookahead — tune: 6=30m, 12=1h, 24=2h, 48=4h

# df_5   = ltf_df.reset_index(drop=True)

high_arr  = df_5['high'].values
low_arr   = df_5['low'].values
open_arr  = df_5['open'].values
close_arr = df_5['close'].values
atr_arr   = df_5['atr42'].values
# ts5_arr   = df_5['ts_5m'].values

labels = np.full(len(df_5), np.nan)

for i in range(len(df_5)):
    if np.isnan(atr_arr[i]):
        continue

    entry = close_arr[i]
    tp    = atr_arr[i] * ATR_MULT
    sl    = atr_arr[i] * ATR_MULT
    label = 0  # default: timeout

    for k in range(1, MAX_BARS_5M + 1):
        j = i + k
        if j >= len(df_5):
            break
        # Stop at data gap — don't evaluate across market closures
        # if ts5_arr[j] != ts5_arr[i] + k * FIVE_MIN_MS:
        #     break

        h, l, o = high_arr[j], low_arr[j], open_arr[j]
        hit_up   = (h - entry) >= tp
        hit_down = (entry - l) >= sl

        if hit_up and hit_down:
            # Both barriers in same bar: open proximity infers direction first
            label = 1 if abs(o - (entry + tp)) < abs(o - (entry - sl)) else -1
            break
        elif hit_up:
            label = 1
            break
        elif hit_down:
            label = -1
            break

    labels[i] = label

df_5['label'] = labels

# Print Stats
total  = df_5['label'].notna().sum()
n_up   = (df_5['label'] ==  1).sum()
n_down = (df_5['label'] == -1).sum()
n_time = (df_5['label'] ==  0).sum()
n_nan  = df_5['label'].isna().sum()

print(f"Total labeled  : {total:,}")
print(f"  Up   ( 1)    : {n_up:,}  ({n_up   / total * 100:.1f}%)")
print(f"  Down (-1)    : {n_down:,}  ({n_down / total * 100:.1f}%)")
print(f"  Timeout (0)  : {n_time:,}  ({n_time  / total * 100:.1f}%)  ← used Two-stage model if this Label=0 dominates") # > 50%
print(f"  NaN (warmup) : {n_nan:,}")

Total labeled  : 546,413
  Up   ( 1)    : 174,148  (31.9%)
  Down (-1)    : 179,672  (32.9%)
  Timeout (0)  : 192,593  (35.2%)  ← used Two-stage model if this Label=0 dominates
  NaN (warmup) : 41


In [5]:
# Drop no use cols
drop_cols = ["close-timestamp", "taker-buy-quotevol","checksum", "quote-vol"]
df_5.drop(columns=drop_cols, inplace=True)

In [6]:
print(df_5.columns)
df_5.head()

Index(['timestamp', 'open', 'high', 'low', 'close', 'vol', 'taker#',
       'taker-buy-basevol', 'atr42', 'label'],
      dtype='object')


,timestamp,open,high,low,close,vol,taker#,taker-buy-basevol,atr42,label
0,1609187400000,26852.179688,26873.220703,26688.109375,26736.640625,598.524536,9027.0,199.124802,NaN,NaN
1,1609187700000,26735.859375,26796.000000,26678.000000,26763.359375,527.531250,9375.0,215.260605,NaN,NaN
2,1609188000000,26763.359375,26824.490234,26733.599609,26779.490234,320.595581,5462.0,131.724228,NaN,NaN
3,1609188300000,26779.480469,26799.919922,26702.000000,26744.679688,254.103180,5055.0,108.755074,NaN,NaN
4,1609188600000,26744.970703,26861.980469,26744.970703,26850.050781,243.299011,4200.0,131.684021,NaN,NaN


# Save Data

In [7]:
# Export marker to json visulize the data on Chart
# save export to jsonl
# Export marker to json visulize the data on Chart
# save export to jsonl

save_path = find_project_root() / "data" / "mlData" / "raw"
save_path.mkdir(parents=True, exist_ok=True)

out_path = save_path / f"{symbol}-{tf}-vX.jsonl"
df_5.to_json(out_path, orient="records", lines=True)
print(f"Saved {len(df_5):,} rows → {out_path}")

Saved 546,454 rows → /Users/aimliu/Library/CloudStorage/OneDrive-Personal/_CODE/Python/py-candle-science/data/mlData/raw/BTCUSDT-5m-vX.jsonl


In [ ]:
print(df_5.columns)